# Chavruta.AI — Embed שו"ת / Responsa (Kaggle GPU)

Embeds the **full Sefaria Responsa library** (גאונים · ראשונים · אחרונים · מודרני) with **bge-m3** (dense + sparse), exactly like the Tanakh / Mishnah / Gemara corpora — so the vectors drop straight into the **same** Qdrant collection.

Input: `shut_chunks.jsonl` (JSONL — one chunk per line, produced by `scripts/fetch_shut.py`).

Settings: Accelerator = **GPU T4 x2 / P100** · Internet = **On**.

Output: `shut_vectors.zip` → `corpus_vectors.npy` + `corpus_sparse.jsonl` + `corpus_meta.jsonl`.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U FlagEmbedding huggingface_hub numpy

In [ ]:
# Pull shut_chunks.jsonl from the HF dataset (same repo as the other corpora).
# First upload it once from your machine:
#   python scripts/upload_dataset_hf.py --repo Yehuda-Rubin/chavruta-torah-mixed --shut
import json
from pathlib import Path
from huggingface_hub import hf_hub_download

HF_DATASET = 'Yehuda-Rubin/chavruta-torah-mixed'
p = hf_hub_download(repo_id=HF_DATASET, filename='shut_chunks.jsonl',
                    repo_type='dataset', local_dir='/kaggle/working')
print('downloaded:', p)

In [ ]:
# JSONL: one chunk per line — never loads the whole corpus into memory at once
jsonl_path = Path('/kaggle/working/shut_chunks.jsonl')

chunks = [json.loads(l) for l in jsonl_path.open(encoding='utf-8')]
docs  = [c['document'] for c in chunks]
ids   = [c['id']       for c in chunks]
metas = [c['metadata'] for c in chunks]
print(f'chunks: {len(chunks):,}')
from collections import Counter
print('by period:', Counter(m.get('period') for m in metas))

In [ ]:
from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device='cuda')
print('model ready')

In [ ]:
import numpy as np, time, pickle

BATCH, MAX_LEN, CKPT_EVERY = 128, 512, 10000
ckpt = Path('/kaggle/working/embed_shut_ckpt.pkl')

if ckpt.exists():
    state = pickle.loads(ckpt.read_bytes())
    dense_parts, sparse_rows, start = state['dense'], state['sparse'], state['done']
    print(f'resuming from {start:,}')
else:
    dense_parts, sparse_rows, start = [], [], 0

t0 = time.time()
for s in range(start, len(docs), BATCH):
    batch = docs[s:s+BATCH]
    enc = model.encode(batch, batch_size=BATCH, max_length=MAX_LEN,
                       return_dense=True, return_sparse=True, return_colbert_vecs=False)
    dense_parts.append(np.asarray(enc['dense_vecs'], dtype='float32'))
    for w in enc['lexical_weights']:
        sparse_rows.append({int(t): float(v) for t, v in dict(w).items()})
    done = min(s + BATCH, len(docs))
    if done % CKPT_EVERY < BATCH or done == len(docs):
        ckpt.write_bytes(pickle.dumps({'dense': dense_parts, 'sparse': sparse_rows, 'done': done}))
    rate = done / max(time.time() - t0, 1e-9)
    eta = (len(docs) - done) / max(rate, 1e-9) / 60
    print(f'  {done:,}/{len(docs):,}  ({rate:.0f}/s, ETA {eta:.0f} min)', end='\r')

vecs = np.vstack(dense_parts)
norms = np.linalg.norm(vecs, axis=1, keepdims=True); norms[norms==0] = 1.0
vecs /= norms
print(f'\ndone: {vecs.shape} in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Write output files — stream to disk, don't hold all in RAM
out = Path('/kaggle/working/out_shut'); out.mkdir(exist_ok=True)

np.save(out / 'corpus_vectors.npy', vecs)
del vecs  # free RAM

with open(out / 'corpus_sparse.jsonl', 'w', encoding='utf-8') as f:
    for i, row in enumerate(sparse_rows):
        f.write(json.dumps({'i': i, 'sparse': row}) + '\n')
del sparse_rows

with open(out / 'corpus_meta.jsonl', 'w', encoding='utf-8') as f:
    for i, (cid, doc, meta) in enumerate(zip(ids, docs, metas)):
        f.write(json.dumps({'i': i, 'id': cid, 'document': doc, 'metadata': meta},
                           ensure_ascii=False) + '\n')

for p in sorted(out.iterdir()):
    print(f'{p.name:28s} {p.stat().st_size/1e6:8.1f} MB')

In [ ]:
# Compress for download
import shutil
archive = '/kaggle/working/shut_vectors'
shutil.make_archive(archive, 'zip', out)
sz = Path(archive + '.zip').stat().st_size / 1e6
print(f'download: {archive}.zip  ({sz:.0f} MB)')

## After download — load into the SAME RAG (keeps Tanakh + Mishnah + Gemara)
```powershell
Expand-Archive shut_vectors.zip -DestinationPath out_shut
# --no-recreate ADDS to the existing collection instead of dropping it
python scripts/load_to_store.py --in out_shut --no-recreate --profile local
# (optional) Hebrew plene/lexical search over the new responsa text:
python scripts/backfill_search_he.py
```